# Camada Bronze — State of Data Brasil (edição 2024)

Dono: Maycon

Responsabilidade desta etapa: ingerir o CSV bruto do `raw` **sem nenhuma limpeza
de conteúdo**, só separando código/descrição de cada coluna e convertendo pra
Parquet particionado por `ano_pesquisa`. Nulo continua nulo, categoria continua
exatamente como veio da pesquisa — a limpeza de verdade acontece na Silver.

**Nota:** o cabeçalho de coluna desta edição segue o mesmo formato
`codigo_descricao` (ex: `2.a_situação_de_trabalho`) que a edição 2025 do Vini —
diferente do formato de tupla Python em texto que a edição 2023 usa. Por isso
reaproveito o parser dele (`extrair_codigo_e_descricao_2025`) em vez de escrever
um novo.

In [1]:
from pyspark.sql import SparkSession, functions as F
import sys
from pathlib import Path

# Procura a raiz do projeto a partir do diretório atual
CAMINHO_ATUAL = Path.cwd().resolve()
for caminho in [CAMINHO_ATUAL, *CAMINHO_ATUAL.parents]:
    if (caminho / "utils" / "config.py").exists():
        RAIZ_PROJETO = caminho
        break
else:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (utils/config.py).")
if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))

from utils.config import CAMINHO_RAW, CAMINHO_BRONZE, CAMINHO_BRONZE_METADADOS
from utils.functions import extrair_codigo_e_descricao_2025

ANO_PESQUISA = 2024

spark = SparkSession.builder.appName(f"state-of-data-{ANO_PESQUISA}-bronze").getOrCreate()
# Importante: Bronze é um caminho ÚNICO particionado por ano_pesquisa, compartilhado
# pelas 3 edições. "dynamic" garante que a escrita só afeta a partição desta edição.
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

print(f"Ano de pesquisa: {ANO_PESQUISA}")
print("Raw:", CAMINHO_RAW)
print("Bronze:", CAMINHO_BRONZE)

c:\Users\Henrique\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Ano de pesquisa: 2024
Raw: C:\Users\Henrique\Desktop\Tech Challenge 3 local\data\raw
Bronze: C:\Users\Henrique\Desktop\Tech Challenge 3 local\data\bronze\state_of_data


In [2]:
CAMINHO_CSV_ORIGEM = CAMINHO_RAW / "state_of_data_2024_2025.csv"

df = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .csv(str(CAMINHO_CSV_ORIGEM))
)

print(f"{df.count()} linhas, {len(df.columns)} colunas")

5217 linhas, 403 colunas


## Separando código e descrição de cada coluna

Cada coluna bruta vem como `codigo_descricao` (ex: `2.a_situação_de_trabalho` →
código `2.a`, descrição `situação_de_trabalho`) — algumas usam `_`, outras usam
espaço como separador, o parser do Vini já trata os dois casos.

In [3]:
pares = [extrair_codigo_e_descricao_2025(c) for c in df.columns]
sem_codigo = [desc for codigo, desc in pares if codigo is None]
print(f"Colunas sem código identificado: {len(sem_codigo)} (esperado: 0)")

# Desambiguação: alguns blocos (ex: 3.f e 4.l) têm opções com a MESMA descrição
# de texto, só o código muda. Sem isso, o rename colide (2 colunas -> 1 nome).
from collections import Counter
contagem = Counter(desc for _, desc in pares)

mapa_codigo_para_nome = {}
for codigo, desc in pares:
    if contagem[desc] > 1:
        # prefixo do código guarda-chuva (ex: "3.f" de "3.f.4") desambigua
        prefixo_bloco = codigo.rsplit(".", 1)[0] if "." in codigo else codigo
        desc = f"{desc} ({prefixo_bloco})"
    mapa_codigo_para_nome[codigo] = desc

duplicadas = [c for c, n in Counter(mapa_codigo_para_nome.values()).items() if n > 1]
print(f"Nomes ainda duplicados após desambiguação: {len(duplicadas)} (esperado: 0)")

for antigo, (codigo, _) in zip(df.columns, pares):
    df = df.withColumnRenamed(antigo, mapa_codigo_para_nome[codigo])

df = df.withColumn("ano_pesquisa", F.lit(ANO_PESQUISA))
print(f"{len(mapa_codigo_para_nome)} códigos mapeados")

Colunas sem código identificado: 0 (esperado: 0)
Nomes ainda duplicados após desambiguação: 0 (esperado: 0)
403 códigos mapeados


## Grava o mapa de colunas (código → nome), usado pela Silver

In [4]:
import csv
import os

CAMINHO_MAPA_COLUNAS = CAMINHO_BRONZE_METADADOS / "mapa_colunas_2024.csv"
os.makedirs(CAMINHO_MAPA_COLUNAS.parent, exist_ok=True)

with open(CAMINHO_MAPA_COLUNAS, "w", encoding="utf-8", newline="") as f:
    escritor = csv.writer(f)
    escritor.writerow(["codigo", "nome_coluna"])
    for codigo, nome_coluna in mapa_codigo_para_nome.items():
        escritor.writerow([codigo, nome_coluna])

print(f"Mapa de colunas salvo em: {CAMINHO_MAPA_COLUNAS} ({len(mapa_codigo_para_nome)} códigos)")

Mapa de colunas salvo em: C:\Users\Henrique\Desktop\Tech Challenge 3 local\data\bronze\metadados\mapa_colunas_2024.csv (403 códigos)


## Grava a Bronze em Parquet

In [5]:
df.write \
    .mode("overwrite") \
    .partitionBy("ano_pesquisa") \
    .parquet(str(CAMINHO_BRONZE))

print(f"Bronze gravada em: {CAMINHO_BRONZE}")

Bronze gravada em: C:\Users\Henrique\Desktop\Tech Challenge 3 local\data\bronze\state_of_data
